# Stochastic Inventory Management Simulation

## Model

The forward state dynamics are
$$
\begin{align}
X_{t + \Delta t} &= X_t + q_{t} \Delta t - (Z_{t + \Delta t} - Z_t)\\
Y_{t + \Delta t} &= Y_t + (-\beta Y_t + \lambda q_t) \Delta t \\
Z_{t + \Delta t} &= Z_t - \theta Z_t \Delta t + \sigma (W_{t+\Delta t} - W_t)
\end{align}
$$

| Symbol | Meaning |
|--------|----------|
| $X$    | Inventory |
| $Y$    | Market impact state |
| $Z$    | Stochastic in-flow |
| $q$    | Trading rate (control) |
| $\theta$ | In-flow mean-reversion speed |
| $\sigma$  | In-flow volatility |
| $\beta, \lambda$ | Impact decay / loading |
| $\varepsilon$ | Spread cost parameter |

## Project structure

```
simulation_project/
├── src/
│   ├── simulation.py       # Core: parameter setup, time grid, path simulation
│   ├── postprocessing.py   # Unit conversion and summary statistics
│   └── plotting.py         # Reusable plot functions
├── experiments/
│   ├── fig1_extreme_paths.py
│   ├── fig2_3_4_theta_sensitivity.py
│   ├── fig5_7_8_11_13_advanced.py
│   └── fig6_9_10_12_parameter_scans.py
└── notebooks/
    └── main.ipynb          ← you are here
```

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import (
    generate_parameters,
    simulate,
    beautify,
    summarize_stats,
    plot_timeseries,
    plot_distributions,
    plot_parameter_scan,
    DEFAULT_COLORS,
    DEFAULT_HUE_ORDER,
)

# Set to 1 for publication-quality plots; 10 for fast experimentation
SPEED_UP_FACTOR = 10

THETAS   = [0.0, 1.0, -1.0]
FLOW_MAP = pd.DataFrame({
    "parameter scans": ["martingale in-flow", "reversal in-flow", "momentum in-flow"],
    "theta": THETAS,
})

## Figure 1 — Two extreme trading paths

In [ ]:
from experiments.fig1_extreme_paths import run_figure1
run_figure1()

## Figures 2, 3, 4 — Sensitivity to θ

In [ ]:
parameters = generate_parameters(theta=THETAS)
summary_stats, timeseries_df, df, N = simulate(
    parameters, nSamples=int(5000 / SPEED_UP_FACTOR)
)
params = parameters.merge(FLOW_MAP, on=["theta"])
stats_output, ts_output = beautify(params, summary_stats, timeseries_df, N)

In [ ]:
# Figure 2 — single representative path
plot_timeseries(ts_output, sample_id=0,
                colors=DEFAULT_COLORS, hue_order=DEFAULT_HUE_ORDER)

In [ ]:
# Figure 3 — cross-sectional mean ± s.d.
plot_timeseries(ts_output, sample_id=None,
                colors=DEFAULT_COLORS, hue_order=DEFAULT_HUE_ORDER)

In [ ]:
# Figure 4 — distribution of key statistics
STATS = [
    "Total variation of in-flow (ADV%)",
    "Internalization (%)",
    "Impact cost per in-flow (bps)",
    "Total variation of out-flow (ADV%)",
    "Internalization regret (%)",
    "Spread cost per in-flow (bps)",
]
plot_distributions(stats_output, stats=STATS,
                   colors=DEFAULT_COLORS, hue_order=DEFAULT_HUE_ORDER)

# Table 2
display(summarize_stats(stats_output).loc[DEFAULT_HUE_ORDER, :])

## Figure 9 — ε sensitivity (deterministic baseline)

In [ ]:
from experiments.fig6_9_10_12_parameter_scans import run_figure9
run_figure9()

## Figure 10 — Joint ε × θ scan

In [ ]:
from experiments.fig6_9_10_12_parameter_scans import run_figure10
run_figure10()

## Figure 6 — Deep σ scan

In [ ]:
from experiments.fig6_9_10_12_parameter_scans import run_figure6
run_figure6()

## Figure 12 — Initial impact state y₀

In [ ]:
from experiments.fig6_9_10_12_parameter_scans import run_figure12
run_figure12()

## Figure 5 — Cost distribution conditional on internalization

In [ ]:
from experiments.fig5_7_8_11_13_advanced import run_figure5
run_figure5()

## Figure 7 — Path monotonicity

In [ ]:
from experiments.fig5_7_8_11_13_advanced import run_figure7
run_figure7()

## Figure 8 — Misspecification costs

In [ ]:
from experiments.fig5_7_8_11_13_advanced import run_figure8
run_figure8()

## Figure 11 — Diffusive limit

In [ ]:
from experiments.fig5_7_8_11_13_advanced import run_figure11
run_figure11()

## Figure 13 — Empirical θ

Requires `empiricalTheta.csv` in the working directory.

In [ ]:
# from experiments.fig5_7_8_11_13_advanced import run_figure13
# run_figure13("../empiricalTheta.csv")